In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/genai/spark_lab"

# List of files to download
files = ["products.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/genai/spark_lab/products.csv', format='csv', header=True)
display(df.limit(10))

### Load the data into a delta table

In [0]:
delta_table_path = f"/Volumes/{catalog_name}/genai/spark_lab/delta/products-delta" 
df.write.format("delta").mode("overwrite").save(delta_table_path)

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Create a deltaTable object
deltaTable = DeltaTable.forPath(spark, delta_table_path)
# Update the table (reduce price of product 771 by 10%)
deltaTable.update(
   condition = "ProductID == 771",
   set = { "ListPrice": "ListPrice * 0.9" })
# View the updated data as a dataframe
deltaTable.toDF().show(10)
     

In [0]:
new_df = spark.read.format("delta").load(delta_table_path)
new_df.show(10)

### Creating a dataframe from the delta dataset

In [0]:
df.write.format("delta").saveAsTable("default.ProductsManaged")

In [0]:
%sql
USE default;
SELECT * FROM ProductsManaged;